# GRAST — WP07.6.2 Curriculum SFT

Chạy trên GPU Colab. Recipe cố định **400 updates copy → 1.200 updates copy+single → 1.200 updates copy+single+multi**. Luôn in log trực tiếp; agent mới verify smoke ngắn, chưa chứng minh curriculum cải thiện reasoning.

Chuẩn bị hai file trong `MyDrive/GRAST/wp07/curriculum/inputs/`:
- `grast-wp07-curriculum-0a891fda1dc3.zip` (data + tokenizer, không chứa weights).
- `sft500-final.pt`: checkpoint SFT 500 updates đã có, **không phải pretrain final.pt hay checkpoint overfit diagnostic**. SHA256 được kiểm trước khi train.

Chọn **Runtime → Change runtime type → GPU**. Lượt mới dùng BF16 nếu GPU hỗ trợ, còn lại FP16. Resume giữ đúng precision/runtime của checkpoint và cùng source pin; đổi GPU/Torch cần bắt đầu một lượt weights-only mới, không gọi là exact resume.


In [ ]:
import hashlib
import importlib.util
import json
import os
import signal
import subprocess
import sys
import tempfile
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
LAST_RUN_DIRECTORY = None


def stream(command, cwd=None):
    global LAST_RUN_DIRECTORY
    print("Running:", " ".join(map(str, command)), flush=True)
    with subprocess.Popen(
        list(map(str, command)),
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    ) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                try:
                    record = json.loads(line)
                except (ValueError, TypeError):
                    continue
                if record.get("event") in ("curriculum_started", "curriculum_finished"):
                    LAST_RUN_DIRECTORY = record["run_directory"]
            returncode = process.wait()
        except KeyboardInterrupt:
            print("Stopping child process; resume from latest committed checkpoint.", flush=True)
            process.send_signal(signal.SIGINT)
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if returncode:
        if LAST_RUN_DIRECTORY:
            print("Last run directory:", LAST_RUN_DIRECTORY, flush=True)
        raise RuntimeError(f"Command failed with exit code {returncode}; inspect logs above.")


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

INPUT_ROOT = Path("/content/drive/MyDrive/GRAST/wp07/curriculum/inputs")
OUTPUT_ROOT = Path("/content/drive/MyDrive/GRAST/wp07/curriculum/runs")
BUNDLE_PATH = INPUT_ROOT / "grast-wp07-curriculum-0a891fda1dc3.zip"
BUNDLE_SHA256 = "02c9e6a9d969365921d00cc119e614295a25f1d1bde53fb7808141f2ffcee217"
INIT_CHECKPOINT = INPUT_ROOT / "sft500-final.pt"
SOURCE_CHECKPOINT_SHA256 = "5f75909261c79e9f7f0c6e37a7e9d76ba5670be8b4be5133b6e148e0cd06ff42"

# Lượt mới: None. Resume: đặt đường dẫn chính xác tới checkpoints/latest.pt.
# Resume chỉ cần file latest.pt và bundle, không cần source/selected/phase-end sibling files.
RESUME_CHECKPOINT = None
# None để chạy đủ 2.800 successful updates; ví dụ 2 cho controlled smoke.
STOP_AFTER = None

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(
        f"Upload curriculum bundle to {BUNDLE_PATH}; it is separate from weights."
    )
if RESUME_CHECKPOINT is not None:
    RESUME_CHECKPOINT = Path(RESUME_CHECKPOINT)
    if not RESUME_CHECKPOINT.is_file():
        raise FileNotFoundError(f"Explicit resume checkpoint does not exist: {RESUME_CHECKPOINT}")
else:
    if not INIT_CHECKPOINT.is_file():
        raise FileNotFoundError(
            f"Missing real SFT500 weights: {INIT_CHECKPOINT}. Copy local "
            "D:/GRAST/artifacts/checkpoints/wp07-cot-sft-math-500/final.pt here. "
            "A summary ZIP does not contain model weights."
        )
    print("Verifying SFT500 weights:", INIT_CHECKPOINT, flush=True)
    actual = sha256(INIT_CHECKPOINT)
    if actual != SOURCE_CHECKPOINT_SHA256:
        raise ValueError(f"Wrong SFT source. Expected {SOURCE_CHECKPOINT_SHA256}; got {actual}")

In [ ]:
WORK_ROOT = Path(tempfile.mkdtemp(prefix="grast-curriculum-"))
SOURCE_ROOT = WORK_ROOT / "source"
SOURCE_COMMIT = "5cb3258b1925f76ed47c9bbcecabc5cb8c9eac22"
stream(
    [
        "git",
        "clone",
        "--branch",
        "feat/wp07-6-2-curriculum-sft",
        "--single-branch",
        "https://github.com/CuongBien/GRAST.git",
        SOURCE_ROOT,
    ]
)
stream(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=SOURCE_ROOT)
# A new subprocess imports the installed source for every command, even if this
# notebook kernel used an older llm_scratch package earlier. PyTorch is not replaced.
stream([sys.executable, "-m", "pip", "install", "--no-deps", SOURCE_ROOT])
if importlib.util.find_spec("matplotlib") is None:
    stream([sys.executable, "-m", "pip", "install", "matplotlib>=3.9,<4"])

DATA_ROOT = WORK_ROOT / "bundle"
stream(
    [
        sys.executable,
        "-u",
        "-m",
        "llm_scratch.curriculum.bundle",
        "extract",
        "--archive",
        BUNDLE_PATH,
        "--output",
        DATA_ROOT,
        "--sha256",
        BUNDLE_SHA256,
    ]
)

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("A GPU runtime is required; select GPU before running training.")
if RESUME_CHECKPOINT is None:
    PRECISION = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
else:
    # Preserve the checkpoint's precision; the runner also validates the complete
    # runtime signature, recipe, implementation, optimizer and data cursor.
    saved = torch.load(RESUME_CHECKPOINT, map_location="cpu", weights_only=True, mmap=True)
    if saved.get("state", {}).get("data_state", {}).get("curriculum_schema_version") != 1:
        raise ValueError(
            "Resume needs a full curriculum latest.pt, not selected_model.pt or old SFT."
        )
    PRECISION = saved["state"]["training_config"]["precision"]
    print("Resume runtime:", saved["state"]["runtime"], flush=True)
    del saved
if PRECISION == "bfloat16" and not torch.cuda.is_bf16_supported():
    raise RuntimeError("This checkpoint requires BF16 hardware for exact resume.")
print(
    json.dumps(
        {
            "gpu": torch.cuda.get_device_name(0),
            "torch": str(torch.__version__),
            "cuda": torch.version.cuda,
            "precision": PRECISION,
            "source_commit": SOURCE_COMMIT,
            "output_root": str(OUTPUT_ROOT),
        }
    ),
    flush=True,
)

## Train hoặc tiếp tục

Log hiển thị ngay khi cập nhật, chuyển stage, validation, sinh đáp án và lưu checkpoint. `latest.pt` được ghi nguyên tử mỗi100 updates; giữ các mốc400/1600/2800. Bộ validation tự sinh cố định48 câu:16 copy,16 single,16 multi. Chọn model bằng macro exact match, hòa điểm giữ mốc sớm hơn. Không dùng test set. Các stage dùng pool tích lũy; không tự chuyển stage theo điểm vừa quan sát.

Khi Colab ngắt, đặt `RESUME_CHECKPOINT` tới **latest.pt** đã lưu và chạy lại notebook với cùng source pin/runtime. `selected_model.pt` là export inference, không dùng để resume optimizer. Dừng giữa một update có thể mất phần từ checkpoint gần nhất; không coi đó là checkpoint giữa microbatch.


In [ ]:
command = [
    sys.executable,
    "-u",
    "-m",
    "llm_scratch.curriculum.runner",
    "--config",
    SOURCE_ROOT / "configs/training/wp07_curriculum.toml",
    "--prepared-root",
    DATA_ROOT / "curriculum_data",
    "--tokenizer",
    DATA_ROOT / "tokenizer",
    "--device",
    "cuda",
    "--precision",
    PRECISION,
    "--output-root",
    OUTPUT_ROOT,
]
if RESUME_CHECKPOINT is None:
    command += ["--init-checkpoint", INIT_CHECKPOINT]
else:
    command += ["--resume", RESUME_CHECKPOINT]
if STOP_AFTER is not None:
    command += ["--stop-after", str(STOP_AFTER)]
stream(command)
print("Run directory:", LAST_RUN_DIRECTORY, flush=True)

In [ ]:
from IPython.display import Image, display

if not LAST_RUN_DIRECTORY:
    raise RuntimeError("No curriculum run started in this notebook session.")
RUN_DIRECTORY = Path(LAST_RUN_DIRECTORY)
report_path = RUN_DIRECTORY / "artifacts/curriculum_report.json"
if report_path.is_file():
    REPORT = json.loads(report_path.read_text())
    print(json.dumps(REPORT, indent=2), flush=True)
    if not REPORT["improved_over_initial"]:
        print(
            "Validation accuracy did not improve; completed training is not a quality pass.",
            flush=True,
        )
for name in ("generated_accuracy.png", "loss_curves.png"):
    image_path = RUN_DIRECTORY / "artifacts" / name
    if image_path.is_file():
        display(Image(filename=str(image_path)))
print("Exact training resume:", RUN_DIRECTORY / "checkpoints/latest.pt", flush=True)
print(
    "Selected model for inference / next evaluation:",
    RUN_DIRECTORY / "checkpoints/selected_model.pt",
    flush=True,
)

In [ ]:
import zipfile

from google.colab import files

SUMMARY_PATH = RUN_DIRECTORY.parent / (RUN_DIRECTORY.name + "-summary.zip")
with zipfile.ZipFile(SUMMARY_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIRECTORY.rglob("*")):
        if path.is_file() and path.suffix not in (".pt", ".pth", ".ckpt"):
            archive.write(path, arcname=str(path.relative_to(RUN_DIRECTORY)))
selected = RUN_DIRECTORY / "checkpoints/selected_model.pt"
if selected.is_file():
    print(
        json.dumps(
            {
                "selected_model": str(selected),
                "size_bytes": selected.stat().st_size,
                "sha256": sha256(selected),
            }
        ),
        flush=True,
    )
print("Summary ZIP (does not contain weights):", SUMMARY_PATH, flush=True)
print(
    "For the next phase, provide selected_model.pt AND this summary ZIP; both remain in Drive.",
    flush=True,
)
files.download(str(SUMMARY_PATH))